# Stage 3 Transformer Classifier - Apache Tika Prototype

This notebook fine-tunes `distilbert-base-uncased` to predict refactoring strategy labels from the Stage 2 `text,label` export.

> **Prototype warning:** This is a Tika-only prototype model, not final evaluation. The current dataset is severely imbalanced and must not be used to make final research claims.

This notebook performs Stage 3 classifier training only. It does not use Nemotron, Google Drive, or Stage 4 backend integration.

## 1. Environment Setup

In [ ]:
# torchvision is not needed for text classification and some Colab images ship an incompatible build.
!pip uninstall -y -q torchvision
!pip install -q transformers datasets evaluate scikit-learn pandas numpy torch joblib accelerate
print("Libraries installed. Restart the runtime once if torchvision was already imported, then run all cells again.")

In [ ]:
import inspect
import json
import os
import random
import shutil
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset, DatasetDict
from google.colab import files
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
OUTPUT_DIR = "./stage3_transformer_output"
SAVE_DIR = Path("saved_transformer_classifier")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Transformers version:", transformers.__version__)
print("PyTorch version:", torch.__version__)

## 2. GPU Check

In [ ]:
CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device("cuda" if CUDA_AVAILABLE else "cpu")
if CUDA_AVAILABLE:
    print("CUDA is available.")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA is unavailable. Training will use CPU and may be slow.")
print("Selected device:", DEVICE)

## 3. Upload And Clean Dataset

Upload the Stage 2 CSV exported by the backend. It must contain `text,label`.

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file was uploaded. Upload a Stage 2 text,label CSV.")

csv_candidates = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_candidates:
    raise ValueError("The uploaded file must have a .csv extension.")
if len(csv_candidates) > 1:
    warnings.warn(f"Multiple CSVs uploaded; using {csv_candidates[0]}.")
CSV_NAME = csv_candidates[0]

try:
    raw_df = pd.read_csv(CSV_NAME)
except Exception as exc:
    raise ValueError(f"Unable to parse {CSV_NAME}: {exc}") from exc

required_columns = {"text", "label"}
missing_columns = required_columns - set(raw_df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

original_rows = len(raw_df)
working_df = raw_df[["text", "label"]].copy()
empty_text_before = int(working_df["text"].fillna("").astype(str).str.strip().eq("").sum())
empty_label_before = int(working_df["label"].fillna("").astype(str).str.strip().eq("").sum())

working_df["text"] = working_df["text"].fillna("").astype(str).str.strip()
working_df["label"] = working_df["label"].fillna("").astype(str).str.strip()
working_df = working_df[(working_df["text"] != "") & (working_df["label"] != "")].copy()
duplicate_pairs = int(working_df.duplicated(subset=["text", "label"]).sum())
working_df = working_df.drop_duplicates(subset=["text", "label"], keep="first").reset_index(drop=True)
df = working_df

if len(df) < 3:
    raise ValueError("At least three valid rows are required.")
if df["label"].nunique() < 2:
    raise ValueError("At least two distinct labels are required for classification.")

print(f"Uploaded file: {CSV_NAME}")
print(f"Original rows: {original_rows}")
print(f"Empty text rows removed: {empty_text_before}")
print(f"Empty label rows removed: {empty_label_before}")
print(f"Duplicate text,label rows removed: {duplicate_pairs}")
print(f"Clean dataset size: {len(df)}")
display(df.head())

## 4. Dataset Quality Analysis

In [ ]:
label_counts = df["label"].value_counts().sort_index()
largest_class = int(label_counts.max())
smallest_class = int(label_counts.min())
imbalance_ratio = float(largest_class / smallest_class)
quality_warnings = []

print("Label counts:")
print(label_counts.to_string())
print(f"\nLargest/smallest imbalance ratio: {imbalance_ratio:.2f}:1")

for label, count in label_counts.items():
    if count < 30:
        message = f"WARNING: Label '{label}' has only {count} examples (recommended minimum: 30)."
        quality_warnings.append(message)
        print(message)
if imbalance_ratio > 3:
    message = f"WARNING: Largest class is {imbalance_ratio:.2f}x the smallest class (recommended maximum: 3x)."
    quality_warnings.append(message)
    print(message)

PROTOTYPE_MESSAGE = "This is a Tika-only prototype model, not final evaluation."
print("\n" + "=" * len(PROTOTYPE_MESSAGE))
print(PROTOTYPE_MESSAGE)
print("=" * len(PROTOTYPE_MESSAGE))

dataset_report = {
    "prototype": True,
    "prototype_message": PROTOTYPE_MESSAGE,
    "source_file": CSV_NAME,
    "original_rows": original_rows,
    "clean_rows": len(df),
    "empty_text_removed": empty_text_before,
    "empty_label_removed": empty_label_before,
    "duplicate_pairs_removed": duplicate_pairs,
    "label_counts": {str(key): int(value) for key, value in label_counts.items()},
    "imbalance_ratio": imbalance_ratio,
    "warnings": quality_warnings,
}

## 5. Label Encoding

In [ ]:
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])
NUM_LABELS = len(label_encoder.classes_)
id2label = {int(index): str(label) for index, label in enumerate(label_encoder.classes_)}
label2id = {label: index for index, label in id2label.items()}

print("Label to ID mapping:")
for label, label_id in label2id.items():
    print(f"  {label} -> {label_id}")

## 6. Train, Validation, And Test Split

The notebook attempts a 70/15/15 stratified split. Tiny classes in the Tika prototype may force a reproducible non-stratified fallback.

In [ ]:
def create_splits(frame):
    try:
        train_frame, temporary_frame = train_test_split(
            frame, test_size=0.30, random_state=SEED, stratify=frame["label_id"]
        )
        validation_frame, test_frame = train_test_split(
            temporary_frame, test_size=0.50, random_state=SEED,
            stratify=temporary_frame["label_id"]
        )
        return train_frame, validation_frame, test_frame, True, None
    except ValueError as exc:
        warning = f"Stratified split was not possible: {exc}. Falling back to non-stratified splitting."
        warnings.warn(warning)
        train_frame, temporary_frame = train_test_split(
            frame, test_size=0.30, random_state=SEED, stratify=None
        )
        validation_frame, test_frame = train_test_split(
            temporary_frame, test_size=0.50, random_state=SEED, stratify=None
        )
        return train_frame, validation_frame, test_frame, False, warning

train_df, validation_df, test_df, used_stratification, split_warning = create_splits(df)
train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

def print_split_distribution(name, frame):
    print(f"\n{name}: {len(frame)} rows ({len(frame) / len(df):.1%})")
    print(frame["label"].value_counts().sort_index().to_string())

print("Stratification used:", used_stratification)
print_split_distribution("Train", train_df)
print_split_distribution("Validation", validation_df)
print_split_distribution("Test", test_df)

dataset_report["split"] = {
    "seed": SEED,
    "requested_ratio": {"train": 0.70, "validation": 0.15, "test": 0.15},
    "sizes": {"train": len(train_df), "validation": len(validation_df), "test": len(test_df)},
    "used_stratification": used_stratification,
    "warning": split_warning,
    "distributions": {
        "train": train_df["label"].value_counts().to_dict(),
        "validation": validation_df["label"].value_counts().to_dict(),
        "test": test_df["label"].value_counts().to_dict(),
    },
}

## 7. Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[["text", "label_id"]], preserve_index=False),
    "validation": Dataset.from_pandas(validation_df[["text", "label_id"]], preserve_index=False),
    "test": Dataset.from_pandas(test_df[["text", "label_id"]], preserve_index=False),
})

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
    )

tokenized_datasets = dataset.map(tokenize_batch, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("label_id", "labels")
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets.set_format("torch")
print(tokenized_datasets)

## 8. Model Setup

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)
print(f"Loaded {MODEL_NAME} with {NUM_LABELS} labels.")

## 9. Metrics

In [ ]:
def compute_metrics(eval_prediction):
    logits, true_labels = eval_prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    predicted_labels = np.argmax(logits, axis=-1)
    precision, recall, weighted_f1, _ = precision_recall_fscore_support(
        true_labels, predicted_labels, average="weighted", zero_division=0
    )
    _, _, macro_f1, _ = precision_recall_fscore_support(
        true_labels, predicted_labels, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "precision": precision,
        "recall": recall,
        "f1": weighted_f1,
        "macro_f1": macro_f1,
    }

## 10. Training

Compatibility logic selects `eval_strategy` in newer Transformers versions and `evaluation_strategy` in older versions.

In [ ]:
training_kwargs = {
    "output_dir": OUTPUT_DIR,
    "save_strategy": "epoch",
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 8,
    "num_train_epochs": 4,
    "weight_decay": 0.01,
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1",
    "greater_is_better": True,
    "save_total_limit": 2,
    "logging_steps": 10,
    "report_to": "none",
    "seed": SEED,
    "data_seed": SEED,
    "fp16": CUDA_AVAILABLE,
}
training_argument_parameters = inspect.signature(TrainingArguments.__init__).parameters
if "eval_strategy" in training_argument_parameters:
    training_kwargs["eval_strategy"] = "epoch"
elif "evaluation_strategy" in training_argument_parameters:
    training_kwargs["evaluation_strategy"] = "epoch"
else:
    raise RuntimeError("Installed Transformers version supports neither eval_strategy nor evaluation_strategy.")

training_args = TrainingArguments(**training_kwargs)
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_datasets["train"],
    "eval_dataset": tokenized_datasets["validation"],
    "compute_metrics": compute_metrics,
}
trainer_parameters = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)
train_result = trainer.train()
print("Training completed.")

## 11. Validation And Test Evaluation

In [ ]:
validation_metrics = trainer.evaluate(
    eval_dataset=tokenized_datasets["validation"],
    metric_key_prefix="validation",
)
test_metrics = trainer.evaluate(
    eval_dataset=tokenized_datasets["test"],
    metric_key_prefix="test",
)
print("Final validation metrics:")
print(json.dumps(validation_metrics, indent=2))
print("\nFinal test metrics:")
print(json.dumps(test_metrics, indent=2))

In [ ]:
test_output = trainer.predict(tokenized_datasets["test"])
test_logits = test_output.predictions[0] if isinstance(test_output.predictions, tuple) else test_output.predictions
test_probabilities = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()
test_predicted_ids = np.argmax(test_probabilities, axis=-1)
test_true_ids = test_output.label_ids

print("Classification report:")
print(classification_report(
    test_true_ids,
    test_predicted_ids,
    labels=list(range(NUM_LABELS)),
    target_names=[id2label[index] for index in range(NUM_LABELS)],
    zero_division=0,
))

confusion = confusion_matrix(
    test_true_ids, test_predicted_ids, labels=list(range(NUM_LABELS))
)
confusion_df = pd.DataFrame(
    confusion,
    index=[f"true_{id2label[index]}" for index in range(NUM_LABELS)],
    columns=[f"pred_{id2label[index]}" for index in range(NUM_LABELS)],
)
print("Confusion matrix:")
display(confusion_df)

## 12. Prediction Function

In [ ]:
def predict_refactoring_category(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("text must be a non-empty string.")
    model.eval()
    model_device = next(model.parameters()).device
    encoded = tokenizer(
        text.strip(),
        return_tensors="pt",
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
    )
    encoded = {key: value.to(model_device) for key, value in encoded.items()}
    with torch.no_grad():
        logits = model(**encoded).logits
        probabilities = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    predicted_id = int(np.argmax(probabilities))
    return {
        "predicted_label": id2label[predicted_id],
        "confidence": float(probabilities[predicted_id]),
        "all_class_probabilities": {
            id2label[index]: float(probability)
            for index, probability in enumerate(probabilities)
        },
    }

In [ ]:
prototype_examples = [
    "This module has duplicated dependency logic shared across many components.",
    "High level module directly depends on low level implementation.",
    "This large component should be extracted into smaller components.",
]
for example in prototype_examples:
    print("\nInput:", example)
    print(json.dumps(predict_refactoring_category(example), indent=2))

## 13. Save Model And Artifacts

In [ ]:
if SAVE_DIR.exists():
    shutil.rmtree(SAVE_DIR)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(SAVE_DIR))
tokenizer.save_pretrained(str(SAVE_DIR))
joblib.dump(label_encoder, SAVE_DIR / "label_encoder.pkl")

label_mapping = {
    "id2label": {str(key): value for key, value in id2label.items()},
    "label2id": label2id,
}
(SAVE_DIR / "label_mapping.json").write_text(
    json.dumps(label_mapping, indent=2), encoding="utf-8"
)

def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    return value

training_metrics = {
    "prototype": True,
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "epochs": 4,
    "train_metrics": train_result.metrics,
    "validation_metrics": validation_metrics,
    "test_metrics": test_metrics,
}
(SAVE_DIR / "training_metrics.json").write_text(
    json.dumps(json_safe(training_metrics), indent=2), encoding="utf-8"
)
(SAVE_DIR / "dataset_report.json").write_text(
    json.dumps(json_safe(dataset_report), indent=2), encoding="utf-8"
)

test_predictions_df = pd.DataFrame({
    "text": test_df["text"].tolist(),
    "true_label": [id2label[int(value)] for value in test_true_ids],
    "predicted_label": [id2label[int(value)] for value in test_predicted_ids],
    "confidence": test_probabilities.max(axis=1),
})
for label_id, label_name in id2label.items():
    test_predictions_df[f"probability_{label_name}"] = test_probabilities[:, label_id]
test_predictions_df.to_csv(SAVE_DIR / "test_predictions.csv", index=False)

readme_text = f"""# DSARP Tika Prototype Transformer Classifier

- Base model: `{MODEL_NAME}`
- Task: refactoring strategy classification
- Labels: {', '.join(label_encoder.classes_)}
- Dataset rows: {len(df)}
- Split: 70% train, 15% validation, 15% test
- Stratified split used: {used_stratification}

## Important limitation

This is a Tika-only prototype model, not final evaluation. The dataset is severely imbalanced and contains rule-derived labels. Metrics must not be interpreted as final research performance.

## Stage 4 location

Copy this complete directory to `backend/app/ml/saved_transformer_classifier/`.
"""
(SAVE_DIR / "README_model.md").write_text(readme_text, encoding="utf-8")

required_artifacts = [
    "label_encoder.pkl",
    "label_mapping.json",
    "training_metrics.json",
    "test_predictions.csv",
    "dataset_report.json",
    "README_model.md",
]
missing_artifacts = [name for name in required_artifacts if not (SAVE_DIR / name).exists()]
if missing_artifacts:
    raise RuntimeError(f"Missing saved artifacts: {missing_artifacts}")
print("Saved artifacts:")
for path in sorted(SAVE_DIR.iterdir()):
    print(" -", path.name)

## 14. Zip And Download

In [ ]:
zip_path = shutil.make_archive(
    "saved_transformer_classifier",
    "zip",
    root_dir=SAVE_DIR.parent,
    base_dir=SAVE_DIR.name,
)
print("Created:", zip_path)
files.download(zip_path)

## 15. Backend Integration Note

This is an instruction only. Stage 4 integration is not performed in this notebook.

In [ ]:
print("Copy saved_transformer_classifier/ into:")
print("backend/app/ml/saved_transformer_classifier/")
print("This will be used in Stage 4 FastAPI model integration.")
print("Stage 4 has not been started by this notebook.")